# Baseline Lead Scoring Models

This notebook trains baseline models on the **LeadForge B2B Lead Scoring** dataset.
It works directly from the pre-generated Parquet files -- no `leadforge` installation required.

We'll cover:
1. Loading the task splits
2. Exploring the features
3. Training Logistic Regression and Gradient Boosting baselines
4. Evaluating with AUC, PR-AUC, and Precision@K
5. Value-aware ranking (probability vs. expected value)
6. Feature importance

**Requirements:** `pandas`, `scikit-learn`, `matplotlib` (all available in Kaggle notebooks by default).

## 1. Load the data

We use the `intermediate` difficulty tier. Change the path to `intro/` or `advanced/` to try other tiers.

In [ ]:
import numpy as np
import pandas as pd

# Adjust this path to your dataset location
BUNDLE = "../intermediate"
TASK = "converted_within_90_days"

train = pd.read_parquet(f"{BUNDLE}/tasks/{TASK}/train.parquet")
valid = pd.read_parquet(f"{BUNDLE}/tasks/{TASK}/valid.parquet")
test = pd.read_parquet(f"{BUNDLE}/tasks/{TASK}/test.parquet")

print(f"Train: {len(train):,} rows")
print(f"Valid: {len(valid):,} rows")
print(f"Test:  {len(test):,} rows")
print("\nConversion rates:")
for name, df in [("train", train), ("valid", valid), ("test", test)]:
    print(f"  {name}: {df[TASK].mean():.1%}")

In [ ]:
# Feature dictionary
feat_dict = pd.read_csv(f"{BUNDLE}/feature_dictionary.csv")
feat_dict

## 2. Explore the features

In [ ]:
# Identify feature types
TARGET = TASK
ID_COLS = ["account_id", "contact_id", "lead_id", "lead_created_at"]
LEAKAGE_COLS = [c for c in train.columns if feat_dict[feat_dict["name"] == c]["leakage_risk"].any()]

print(f"Leakage-flagged columns (excluded): {LEAKAGE_COLS}")

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET] + LEAKAGE_COLS]
cat_cols = [c for c in feature_cols if train[c].dtype == "string" or train[c].dtype == "object"]
bool_cols = [c for c in feature_cols if train[c].dtype == "boolean"]
num_cols = [c for c in feature_cols if c not in cat_cols + bool_cols]

print(f"\nCategorical: {len(cat_cols)} -- {cat_cols}")
print(f"Boolean:     {len(bool_cols)} -- {bool_cols}")
print(f"Numeric:     {len(num_cols)} -- {num_cols}")

In [ ]:
# Missing values
missing = train[feature_cols].isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Missing values in training set:")
    for col, count in missing.items():
        print(f"  {col}: {count} ({count / len(train):.1%})")
else:
    print("No missing values in training set.")

In [ ]:
# Summary statistics for numeric features
train[num_cols].describe().T

## 3. Build preprocessing pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Convert boolean columns to int for sklearn
def prep_df(df):
    out = df[feature_cols].copy()
    for c in bool_cols:
        out[c] = out[c].astype("Int64")
    return out


numeric_features = num_cols + bool_cols
categorical_features = cat_cols

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_features,
        ),
    ]
)

X_train = prep_df(train)
y_train = train[TARGET].astype(int)
X_test = prep_df(test)
y_test = test[TARGET].astype(int)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 4. Train baselines and evaluate

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, solver="lbfgs", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
}

results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    # Precision@K
    for k in [25, 50, 100]:
        top_k_idx = np.argsort(-y_prob)[:k]
        p_at_k = y_test.iloc[top_k_idx].mean()
        base_rate = y_test.mean()
        lift = p_at_k / base_rate
        results.append(
            {
                "Model": name,
                "Metric": f"P@{k}",
                "Value": f"{p_at_k:.3f}",
                "Lift": f"{lift:.2f}x",
            }
        )

    results.append({"Model": name, "Metric": "ROC-AUC", "Value": f"{auc:.3f}", "Lift": ""})
    results.append({"Model": name, "Metric": "PR-AUC", "Value": f"{pr_auc:.3f}", "Lift": ""})
    fitted_models[name] = pipe
    print(f"{name}: AUC={auc:.3f}, PR-AUC={pr_auc:.3f}")

pd.DataFrame(results)

## 5. Value-aware ranking

When deals have different sizes (`expected_acv`), ranking by probability alone leaves money on the table.
Ranking by expected value (P(convert) x ACV) captures more revenue in the top-K.

In [ ]:
if "expected_acv" in test.columns:
    best_pipe = fitted_models["Gradient Boosting"]
    y_prob = best_pipe.predict_proba(X_test)[:, 1]
    acv = test["expected_acv"].fillna(test["expected_acv"].median()).values
    ev = y_prob * acv

    true_acv = test["expected_acv"].fillna(0).values
    converted = y_test.values.astype(bool)

    for k in [25, 50, 100]:
        # Probability ranking
        prob_top_k = np.argsort(-y_prob)[:k]
        prob_acv = true_acv[prob_top_k][converted[prob_top_k]].sum()

        # EV ranking
        ev_top_k = np.argsort(-ev)[:k]
        ev_acv = true_acv[ev_top_k][converted[ev_top_k]].sum()

        uplift = (ev_acv - prob_acv) / prob_acv * 100 if prob_acv > 0 else 0
        print(
            f"K={k}: Prob ranking ${prob_acv:,.0f} | "
            f"EV ranking ${ev_acv:,.0f} | Uplift: {uplift:+.1f}%"
        )
else:
    print("No expected_acv column found.")

## 6. Feature importance (Gradient Boosting)

In [ ]:
import matplotlib.pyplot as plt

gbm_pipe = fitted_models["Gradient Boosting"]
gbm_model = gbm_pipe.named_steps["model"]
preproc = gbm_pipe.named_steps["preprocess"]

# Get feature names after encoding
num_names = numeric_features
cat_encoder = preproc.named_transformers_["cat"].named_steps["encoder"]
cat_names = list(cat_encoder.get_feature_names_out(categorical_features))
all_names = num_names + cat_names

importances = gbm_model.feature_importances_
feat_imp = pd.Series(importances, index=all_names).sort_values(ascending=False)

top_n = 15
fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.head(top_n).plot.barh(ax=ax)
ax.set_xlabel("Importance")
ax.set_title(f"Top {top_n} Features (Gradient Boosting)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTop {top_n} features:")
for name, imp in feat_imp.head(top_n).items():
    print(f"  {name}: {imp:.4f}")

## 7. Try other difficulty tiers

Change `BUNDLE` at the top of this notebook to point to `intro/` or `advanced/` and re-run all cells.
You should see:
- **Intro:** Higher AUC (cleaner signal, ~2% missing values)
- **Intermediate:** Moderate AUC (~8% missing values, more noise)
- **Advanced:** Lower AUC (~18% missing values, much noisier)

## Explore the relational tables

The flat task splits are derived from 9 relational tables under `tables/`. You can engineer your own features:

```python
touches = pd.read_parquet(f"{BUNDLE}/tables/touches.parquet")
sessions = pd.read_parquet(f"{BUNDLE}/tables/sessions.parquet")
# ... join, aggregate, and build features from raw events
```

See the [leadforge README](https://github.com/leadforge-dev/leadforge) for more details.